# 04. Content-Based Recommender

## Objective

The goal of this notebook is to build a content-based movie recommendation
system using the processed MovieLens metadata.

The recommender will identify movies that are similar based on their
content, including:

* Movie genres
* User-generated tags

We will:

1. Load the processed movie content dataset.
2. Prepare the combined content text.
3. Convert movie content into TF-IDF vectors.
4. Calculate similarity between movies using cosine similarity.
5. Build a function that returns similar movies for a selected movie.
6. Evaluate the quality of the recommendations using example movies.

The resulting recommender will provide the first recommendation capability
for the overall hybrid movie recommendation system.


In [2]:
# Import the libraries needed for content-based recommendation
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
# Define the path to the processed movie content dataset
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed data:", PROCESSED_DIR)

Processed data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed


In [4]:
# Load the processed movie content dataset
movies_content = pd.read_csv(
    PROCESSED_DIR / "movies_content.csv"
)

print("Dataset loaded successfully.")
print("Shape:", movies_content.shape)

Dataset loaded successfully.
Shape: (87585, 6)


In [5]:
# Inspect the columns and first few movie records
print(movies_content.columns.tolist())

movies_content.head()

['movieId', 'title', 'genres_clean', 'release_year', 'tags_text', 'content_text']


,movieId,title,genres_clean,release_year,tags_text,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,children disney animation children disney disn...,adventure animation children comedy fantasy ch...
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,robin williams fantasy robin williams time tra...,adventure children fantasy robin williams fant...
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,comedinha de velhinhos engraãƒâ§ada comedinha ...,comedy romance comedinha de velhinhos engraãƒâ...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,characters slurs based on novel or book chick ...,comedy drama romance characters slurs based on...
4,5,Father of the Bride Part II (1995),Comedy,1995.0,fantasy pregnancy remake family steve martin s...,comedy fantasy pregnancy remake family steve m...


In [6]:
# Import the TF-IDF vectorizer from scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:
# Configure how TF-IDF will convert movie text into numerical features
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

In [8]:
# Transform movie content text into TF-IDF feature vectors
tfidf_matrix = tfidf.fit_transform(
    movies_content["content_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (87585, 248779)


# Check the sparsity: Let's quantify just how sparse it is before calculating similarities.

In [9]:
# Measure how sparse the TF-IDF matrix is
total_values = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
nonzero_values = tfidf_matrix.nnz

sparsity = 1 - (nonzero_values / total_values)

print("Non-zero values:", nonzero_values)
print("Sparsity:", round(sparsity * 100, 2), "%")

Non-zero values: 2911018
Sparsity: 99.99 %


# Calculate movie similarity

In [10]:
# Import the function used to calculate similarity between movie vectors
from sklearn.metrics.pairwise import cosine_similarity

# Test similarity for one movie

We do not want to calculate similarity between all 87,585 movies at once. That would create an enormous matrix. Instead, we'll first choose one movie and compare it with all other movies.
Let's use movieId = 1 as a test. In MovieLens, this is Toy Story (1995).

In [11]:
# Find the row position of the selected movie in the TF-IDF matrix
movie_id = 1

movie_index = movies_content.index[
    movies_content["movieId"] == movie_id
][0]

print("Movie index:", movie_index)
print("Movie title:", movies_content.loc[movie_index, "title"])

Movie index: 0
Movie title: Toy Story (1995)


In [13]:
# Calculate how similar Toy Story is to every movie in the dataset
movie_vector = tfidf_matrix[movie_index]

similarities = cosine_similarity(
    movie_vector,
    tfidf_matrix
)

print("Similarity matrix shape:", similarities.shape)

Similarity matrix shape: (1, 87585)


In [14]:
# Sort movies by similarity score and select the top 10 matches
similarity_scores = similarities[0]

top_indices = similarity_scores.argsort()[::-1][1:11]

movies_content.iloc[top_indices][
    ["movieId", "title", "genres_clean"]
]

,movieId,title,genres_clean
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
14815,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
4781,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6259,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy
8248,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy
39850,157296,Finding Dory (2016),Adventure|Animation|Comedy
11363,50872,Ratatouille (2007),Animation|Children|Drama
26634,126405,The Adventures of André and Wally B. (1984),Animation
10812,45517,Cars (2006),Animation|Children|Comedy


In [15]:
# Create a table showing each recommended movie and its similarity score
recommendations = movies_content.iloc[top_indices][
    ["movieId", "title", "genres_clean"]
].copy()

recommendations["similarity_score"] = similarity_scores[top_indices]

recommendations

,movieId,title,genres_clean,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.823005
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.702529
14815,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.599533
4781,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.575633
6259,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.562250
8248,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.555139
39850,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.526530
11363,50872,Ratatouille (2007),Animation|Children|Drama,0.520898
26634,126405,The Adventures of André and Wally B. (1984),Animation,0.513514
10812,45517,Cars (2006),Animation|Children|Comedy,0.496136


# Create the recommendation function

In [22]:
# Check how many movies have no TF-IDF features
zero_content_movies = (tfidf_matrix.getnnz(axis=1) == 0).sum()

print("Movies with no TF-IDF features:", zero_content_movies)

Movies with no TF-IDF features: 4426


In [23]:
# Create a safer content-based recommendation function
def recommend_movies(movie_id, n=10):
    
    # Check whether the movie exists
    matching_rows = movies_content.index[
        movies_content["movieId"] == movie_id
    ]

    if len(matching_rows) == 0:
        return "Movie ID not found."

    movie_index = matching_rows[0]

    # Check whether the movie has usable TF-IDF features
    if tfidf_matrix[movie_index].nnz == 0:
        return "This movie does not have enough content information for content-based recommendations."

    # Calculate similarity with all movies
    movie_vector = tfidf_matrix[movie_index]

    similarities = cosine_similarity(
        movie_vector,
        tfidf_matrix
    )[0]

    # Select the most similar movies
    top_indices = similarities.argsort()[::-1][1:n+1]

    recommendations = movies_content.iloc[top_indices][
        ["movieId", "title", "genres_clean"]
    ].copy()

    recommendations["similarity_score"] = similarities[top_indices]

    return recommendations

In [24]:
# Test the recommendation function using Toy Story
recommend_movies(movie_id=1, n=5)

,movieId,title,genres_clean,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.823005
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.702529
14815,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.599533
4781,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.575633
6259,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.562250


In [25]:
# Test how the recommender handles a movie ID that does not exist
recommend_movies(movie_id=999999999, n=5)

'Movie ID not found.'

In [26]:
# Find a movie that has no usable TF-IDF features
zero_content_index = np.where(
    tfidf_matrix.getnnz(axis=1) == 0
)[0][0]

zero_content_movie_id = movies_content.iloc[
    zero_content_index
]["movieId"]

zero_content_title = movies_content.iloc[
    zero_content_index
]["title"]

print("Movie ID:", zero_content_movie_id)
print("Movie title:", zero_content_title)

Movie ID: 91246
Movie title: Milky Way (Tejút) (2007)


In [28]:
# Test how the recommender handles a movie with no usable content
recommend_movies(movie_id=91246, n=5)

'This movie does not have enough content information for content-based recommendations.'

# Both safety checks work:

Invalid movie ID → handled correctly.
Movie with no content → handled correctly.
Normal movie → recommendations work correctly.

In [29]:
# Test the content-based recommender with a different movie
recommend_movies(movie_id=318, n=10)

,movieId,title,genres_clean,similarity_score
46,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,0.464554
10722,44665,Lucky Number Slevin (2006),Crime|Drama|Mystery,0.384548
2670,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,0.362719
11215,49284,10 Items or Less (2006),Comedy|Drama|Romance,0.347095
49,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,0.337701
8071,8783,"Village, The (2004)",Drama|Mystery|Thriller,0.333820
2867,2959,Fight Club (1999),Action|Crime|Drama|Thriller,0.331035
18559,96969,"Magic of Belle Isle, The (2012)",Comedy|Drama,0.317630
1873,1962,Driving Miss Daisy (1989),Drama,0.313789
12007,56587,"Bucket List, The (2007)",Comedy|Drama,0.307848


# Our content-based recommender now has:

* Cleaned movie genres
* Aggregated user tags
* Combined genres + tags
* TF-IDF representation
* Sparse matrix handling
* Cosine similarity
* Similar-movie recommendations
* Similarity scores
* Reusable recommendation function
* Invalid movie-ID handling
* Missing-content handling
* Testing with animation, drama/crime, and action/sci-fi movies


# save the trained TF-IDF objects

In [31]:
# Import the function used to save the sparse TF-IDF matrix
from scipy.sparse import save_npz

In [32]:
# Save the sparse TF-IDF matrix for reuse in later notebooks and the app
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_npz(
    MODELS_DIR / "tfidf_matrix.npz",
    tfidf_matrix
)

print("TF-IDF matrix saved successfully.")

TF-IDF matrix saved successfully.


# Save the TF-IDF vectorizer

In [33]:
# Import joblib so the fitted TF-IDF vectorizer can be saved
import joblib

In [34]:
# Save the fitted TF-IDF vectorizer for reuse in later notebooks and the app
joblib.dump(
    tfidf,
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

print("TF-IDF vectorizer saved successfully.")

TF-IDF vectorizer saved successfully.


## Conclusion

The content-based movie recommender was successfully developed using genres and user-generated tags.

TF-IDF was used to represent movie content, and cosine similarity was used to find similar movies. The recommender was tested with several movies and produced reasonable results.

The trained TF-IDF matrix and vectorizer were also saved for future use.

### Next Step

The next notebook will develop a **collaborative filtering recommender** using user-rating behavior to generate personalized movie recommendations.
